In [1]:
#%pip install pypdf
#%pip install scikit-learn

In [2]:
import json
from pathlib import Path
from urllib.request import urlretrieve

from pypdf import PdfReader

budget_url = "https://budget.canada.ca/2026/report-rapport/pdf/budget-2026.pdf"
pdf_path = Path("budget-2026.pdf")
output_path = Path("budget_pages.jsonl")

if not pdf_path.exists():
    urlretrieve(budget_url, pdf_path)

reader = PdfReader(pdf_path)

with output_path.open("w", encoding="utf-8") as output:
    for page_number, page in enumerate(reader.pages, start=1):
        record = {
            "page": page_number,
            "text": page.extract_text() or "",
            "source": f"{budget_url}#page={page_number}",
        }
        output.write(json.dumps(record, ensure_ascii=False) + "\n")

print(f"Saved {len(reader.pages)} PDF pages to {output_path}")

Saved 532 PDF pages to budget_pages.jsonl


In [3]:
#Indexed 1,097 passages.

In [4]:
import json
import re
from bisect import bisect_right
from pathlib import Path

from sklearn.feature_extraction.text import TfidfVectorizer

chunk_size = 250
overlap = 50
passages = []

sentence_boundary = re.compile(r"""[.!?]["'”’)\]]*\s+(?=[A-Z0-9“"‘'(])""")

with Path("budget_pages.jsonl").open(encoding="utf-8") as file:
    pages = [json.loads(line) for line in file if line.strip()]

page_runs = []
for page in sorted(pages, key=lambda page: int(page["page"])):
    text = " ".join(page["text"].split())
    if not text:
        continue

    page_number = int(page["page"])
    if not page_runs or page_number != page_runs[-1][-1][0] + 1:
        page_runs.append([])

    page_runs[-1].append((page_number, page["source"], text))

for run in page_runs:
    page_starts = []
    offset = 0
    for _, _, text in run:
        page_starts.append(offset)
        offset += len(text) + 1

    text = " ".join(page_text for _, _, page_text in run)

    sentence_starts = [0]
    sentence_starts.extend(match.end() for match in sentence_boundary.finditer(text))
    sentence_starts.append(len(text))

    word_counts = [
        len(text[start:end].split())
        for start, end in zip(sentence_starts, sentence_starts[1:])
    ]

    start = 0
    while start < len(word_counts):
        end = start + 1
        total_words = word_counts[start]

        while end < len(word_counts) and total_words + word_counts[end] <= chunk_size:
            total_words += word_counts[end]
            end += 1

        chunk_start = sentence_starts[start]
        excerpt = text[chunk_start:sentence_starts[end]].rstrip()
        first = bisect_right(page_starts, chunk_start) - 1
        last = bisect_right(page_starts, chunk_start + len(excerpt) - 1) - 1

        first_page = run[first][0]
        last_page = run[last][0]
        passages.append({
            "text": excerpt,
            "page": first_page if first == last else f"{first_page}–{last_page}",
            "source": run[first][1],
            "sources": [source for _, source, _ in run[first:last + 1]],
        })

        if end == len(word_counts):
            break

        next_start = end
        overlap_words = 0
        while next_start > start + 1:
            sentence_words = word_counts[next_start - 1]
            if overlap_words + sentence_words > overlap:
                break
            if overlap_words + sentence_words + word_counts[end] > chunk_size:
                break
            overlap_words += sentence_words
            next_start -= 1

        start = next_start

if not passages:
    raise ValueError("The JSONL file contains no extracted text.")

vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2))
search_index = vectorizer.fit_transform([passage["text"] for passage in passages])
print(f"Indexed {len(passages):,} passages.")

Indexed 958 passages.


In [7]:
from sklearn.metrics.pairwise import cosine_similarity

#question = "What housing measures does the federal budget propose?"
question = "What timeline does the budget give for ending homelessness?"



question_vector = vectorizer.transform([question])
scores = cosine_similarity(question_vector, search_index).ravel()

matches = []
for index in scores.argsort()[::-1]:
    if scores[index] <= 0:
        break
    matches.append(passages[index])

if not matches:
    print("No keyword matches. Try a more specific question.")

for rank, passage in enumerate(matches, start=1):
    print(f"\n{rank}. PDF page {passage['page']}")
    print(passage["source"])
    print(passage["text"])


1. PDF page 13–14
https://budget.canada.ca/2026/report-rapport/pdf/budget-2026.pdf#page=13
This includes jobs, training, and income supports • Housing: As part of this plan the Government of Canada is making a historic, multi-billion-dollar investment to build the transitional and permanent housing necessary to pragmatically ending the homelessness crisis • Health: for those who need it, we will build the facilities needed to care for and support the most vulnerable in our society. We will begin immediately in Winnipeg to break the cycle of homelessness and provide the care and permanent housing for the more than 2,000 people who are currently homeless in that city. We will work with public and private partners to build a new Centre for Mental Health and Addictions and add an additional 2,000 new transitional and supportive homes. With an estimated 65,000 homeless Canadians, ending homelessness must become a national initiative. Everyone must play their role: the federal government by

In [9]:
import json
from urllib.request import Request, urlopen

top_matches = matches[:5]

if top_matches:
    excerpts = "\n\n".join(
        f"PDF page {match['page']}\n{match['text']}"
        for match in top_matches
    )

    payload = {
        "model": "qwen2.5:3b",
        "stream": False,
        "messages": [
            {
                "role": "system",
                "content": (
                    "Answer using only the supplied budget excerpts. "
                    "Cite the PDF page for each claim. "
                    "If the excerpts lack enough evidence, say so. "
                    "For timeline questions, answer in one sentence and quote the timeline exactly as written. "
                    "Do not convert a relative timeline into a calendar year or fiscal year."

                    
                ),
            },
            {
                "role": "user",
                "content": f"Question: {question}\n\nBudget excerpts:\n{excerpts}",
            },
        ],
    }

    request = Request(
        "http://127.0.0.1:11434/api/chat",
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST",
    )

    with urlopen(request, timeout=180) as response:
        answer = json.load(response)["message"]["content"]

    print(f"QUESTION\n{question}")
    print(f"\nQWEN'S ANSWER\n{answer}")
    print("\nPDF EXCERPTS SENT TO QWEN")

    for number, match in enumerate(top_matches, start=1):
        print(f"\nExcerpt {number} | PDF page {match['page']}")
        print(match["source"])
        print(match["text"])
else:
    print("No matching passages. Try a different question in the search cell.")

QUESTION
What timeline does the budget give for ending homelessness?

QWEN'S ANSWER
The budget proposes to end homelessness within 10 years, which starts in 2027-28. This is based on the timeline provided in the Budget 2026, as stated on page 212.

PDF EXCERPTS SENT TO QWEN

Excerpt 1 | PDF page 13–14
https://budget.canada.ca/2026/report-rapport/pdf/budget-2026.pdf#page=13
This includes jobs, training, and income supports • Housing: As part of this plan the Government of Canada is making a historic, multi-billion-dollar investment to build the transitional and permanent housing necessary to pragmatically ending the homelessness crisis • Health: for those who need it, we will build the facilities needed to care for and support the most vulnerable in our society. We will begin immediately in Winnipeg to break the cycle of homelessness and provide the care and permanent housing for the more than 2,000 people who are currently homeless in that city. We will work with public and private par

In [11]:
import json
import tkinter as tk
from tkinter import ttk
from concurrent.futures import ThreadPoolExecutor
from urllib.request import Request, urlopen

from sklearn.metrics.pairwise import cosine_similarity

MODEL = "qwen2.5:3b"  # Replace with the model name in your working Qwen cell.
OLLAMA_URL = "http://localhost:11434/api/chat"

SYSTEM_PROMPT = (
    "Answer directly using only the supplied budget excerpts. "
    "Cite the PDF page for each claim. "
    "If the excerpts lack enough evidence, say so. "
    "For timeline questions, answer in one sentence and quote the timeline "
    "exactly as written. Do not convert it into a calendar or fiscal year."
)


def ask_budget(question):
    question_vector = vectorizer.transform([question])
    scores = cosine_similarity(question_vector, search_index).ravel()
    matches = [
        passages[index]
        for index in scores.argsort()[::-1][:5]
        if scores[index] > 0
    ]

    if not matches:
        return "I found no matching budget excerpts. Try a more specific question."

    excerpts = "\n\n".join(
        f"PDF page {passage['page']}\n{passage['text']}"
        for passage in matches
    )

    payload = {
        "model": MODEL,
        "stream": False,
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Question: {question}\n\n{excerpts}"},
        ],
    }
    request = Request(
        OLLAMA_URL,
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"},
    )
    with urlopen(request, timeout=120) as response:
        return json.load(response)["message"]["content"].strip()


def show_answer(text):
    answer.configure(state="normal")
    answer.delete("1.0", "end")
    answer.insert("1.0", text)
    answer.configure(state="disabled")


def submit(event=None):
    question = question_box.get().strip()
    if not question or ask_button.instate(["disabled"]):
        return

    ask_button.state(["disabled"])
    show_answer("Searching the budget…")
    future = worker.submit(ask_budget, question)
    window.after(100, check_result, future)


def check_result(future):
    if not future.done():
        window.after(100, check_result, future)
        return

    try:
        show_answer(future.result())
    except Exception as error:
        show_answer(f"Request failed. Check Ollama and the model name.\n\n{error}")
    finally:
        ask_button.state(["!disabled"])


window = tk.Tk()
window.title("Eric Summer TV | Canada Budget 2026 Chatbot")
window.geometry("640x360")
window.minsize(420, 280)

worker = ThreadPoolExecutor(max_workers=1)

panel = ttk.Frame(window, padding=20)
panel.pack(fill="both", expand=True)

ttk.Label(panel, text="Eric Summer TV").pack(anchor="w")
ttk.Label(
    panel,
    text="Canada Budget 2026 Chatbot",
    font=("Segoe UI", 18, "bold"),
).pack(anchor="w", pady=(4, 16))

question_row = ttk.Frame(panel)
question_row.pack(fill="x")

question_box = ttk.Entry(question_row)
question_box.pack(side="left", fill="x", expand=True)
question_box.bind("<Return>", submit)
question_box.focus_set()

ask_button = ttk.Button(question_row, text="Ask", command=submit)
ask_button.pack(side="left", padx=(8, 0))

answer = tk.Text(panel, wrap="word", state="disabled", font=("Segoe UI", 11))
answer.pack(fill="both", expand=True, pady=(12, 0))

try:
    window.mainloop()
finally:
    worker.shutdown(wait=False, cancel_futures=True)